<a href="https://colab.research.google.com/github/charlyacha/labo1-colabs/blob/main/05_Cuadrados_minimos_y_el_coeficiente_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Colab 05 — Cuadrados mínimos, residuos y el coeficiente de correlación R**Laboratorio 1 · Clase 5****Objetivos.**1. Ajustar una recta por cuadrados mínimos y obtener **la incerteza de los parámetros**.2. Incorporar el **gráfico de residuos** como parte obligatoria de todo ajuste, desde hoy y para   siempre.3. Entender qué mide —y sobre todo qué **no** mide— el coeficiente de correlación $R$.**Requisitos previos:** Colabs 01 a 04.> **Antes de empezar:** hacé `Archivo → Guardar una copia en Drive`. Vas a trabajar sobre *tu* copia; el original queda intacto para el resto del curso.

In [ ]:
import numpy as npimport matplotlib.pyplot as pltfrom scipy.optimize import curve_fitnp.random.seed(20260909)

---## 1. El criterioTenemos pares $(x_i, y_i)$ y proponemos el modelo $y = a x + b$. Cuadrados mínimos elige $a$ y $b$que minimizan$$ S(a,b) = \\sum_{i=1}^{N} \\left[y_i - (a x_i + b)\\right]^2 $$Dos cosas que conviene tener claras antes de apretar el botón:- El criterio supone que **el error está en $y$** y que $x$ es conocida con precisión mucho mayor.  Si no es tu caso, cuadrados mínimos ordinarios no es la herramienta correcta.- Supone además que **todos los puntos tienen la misma incerteza**. Casi nunca es cierto en un  laboratorio real; lo arreglamos en el Colab 06.Derivando $S$ respecto de $a$ y $b$ e igualando a cero se llega a expresiones cerradas. Lasimplementamos una vez, a mano, para que no sea una caja negra.

In [ ]:
def cuadrados_minimos(x, y):    """Ajuste lineal no ponderado. Devuelve (a, b, sigma_a, sigma_b, s_y)."""    x, y = np.asarray(x, float), np.asarray(y, float)    N = len(x)    Sx, Sy = x.sum(), y.sum()    Sxx, Sxy = (x*x).sum(), (x*y).sum()    Delta = N*Sxx - Sx**2    a = (N*Sxy - Sx*Sy) / Delta          # pendiente    b = (Sxx*Sy - Sx*Sxy) / Delta        # ordenada al origen    # dispersión de los datos alrededor de la recta (N-2: dos parámetros ajustados)    residuos = y - (a*x + b)    s_y = np.sqrt((residuos**2).sum() / (N - 2))    sigma_a = s_y * np.sqrt(N / Delta)    sigma_b = s_y * np.sqrt(Sxx / Delta)    return a, b, sigma_a, sigma_b, s_y

In [ ]:
# --- DATOS DE EJEMPLO: ley de Hooke (reemplazar por los propios) ---masa   = np.array([0.050, 0.100, 0.150, 0.200, 0.250, 0.300, 0.350, 0.400])   # kgelong  = np.array([4.05, 8.20, 12.05, 16.40, 20.15, 24.55, 28.30, 32.60])     # cm# -------------------------------------------------------------------a, b, da, db, s_y = cuadrados_minimos(masa, elong)print(f"pendiente a = {a:.3f} ± {da:.3f} cm/kg")print(f"ordenada  b = {b:.3f} ± {db:.3f} cm")print(f"dispersión de los datos alrededor de la recta: s_y = {s_y:.3f} cm")# de la pendiente sale la constante del resorte: Δx = (g/k) m  =>  k = g/ag = 9.81k = g / (a/100)                      # a está en cm/kg -> pasamos a m/kgdk = k * (da/a)print(f"\nk = {k:.1f} ± {dk:.1f} N/m")

El mismo ajuste con `curve_fit`, que es lo que vamos a usar de acá en adelante porque generaliza acualquier modelo:

In [ ]:
def recta(x, a, b):    return a*x + bpopt, pcov = curve_fit(recta, masa, elong)perr = np.sqrt(np.diag(pcov))print("curve_fit :", f"a = {popt[0]:.3f} ± {perr[0]:.3f}, b = {popt[1]:.3f} ± {perr[1]:.3f}")print("a mano    :", f"a = {a:.3f} ± {da:.3f}, b = {b:.3f} ± {db:.3f}")

Coinciden, como tiene que ser. `curve_fit` sin el argumento `sigma` supone que todos los puntospesan igual y **estima** la dispersión a partir de los propios residuos — exactamente lo que hacenuestra función. En el Colab 06 vamos a ver por qué eso, en un experimento real, casi siempre estámal.

---## 2. El gráfico de residuosEl residuo del punto $i$ es$$ r_i = y_i - f(x_i) $$Si el modelo es adecuado, los residuos deben verse como ruido alrededor de cero: **sin estructura,sin tendencia, sin curvatura**. Cualquier patrón visible es el modelo diciéndote que le falta algo.En la práctica, mirar los residuos es más informativo que cualquier estadístico único. Por eso, deacá en adelante, **todo ajuste de este curso se grafica junto con sus residuos**.

In [ ]:
def grafico_con_residuos(x, y, modelo, popt, yerr=None,                         xlabel='x', ylabel='y', titulo='', etiqueta_modelo='ajuste'):    """Figura estándar del curso: datos + modelo arriba, residuos abajo."""    x, y = np.asarray(x, float), np.asarray(y, float)    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 5.6), sharex=True,                                   gridspec_kw={'height_ratios': [3, 1]})    xx = np.linspace(x.min(), x.max(), 400)    if yerr is None:        ax1.plot(x, y, 'o', ms=5, label='datos')    else:        ax1.errorbar(x, y, yerr=yerr, fmt='o', ms=5, capsize=3, label='datos')    ax1.plot(xx, modelo(xx, *popt), 'crimson', lw=1.8, label=etiqueta_modelo)    ax1.set_ylabel(ylabel); ax1.grid(alpha=0.3); ax1.legend()    if titulo: ax1.set_title(titulo)    res = y - modelo(x, *popt)    if yerr is None:        ax2.plot(x, res, 'o', ms=5)    else:        ax2.errorbar(x, res, yerr=yerr, fmt='o', ms=5, capsize=3)    ax2.axhline(0, color='crimson', lw=1.2)    ax2.set_xlabel(xlabel); ax2.set_ylabel('residuo')    ax2.grid(alpha=0.3)    fig.subplots_adjust(hspace=0.08, left=0.13, right=0.97, top=0.93, bottom=0.11)    return fig, (ax1, ax2)grafico_con_residuos(masa, elong, recta, popt,                     xlabel='Masa $m$ [kg]', ylabel='Elongación $\\Delta x$ [cm]',                     titulo='Ley de Hooke', etiqueta_modelo='$\\Delta x = a\\,m + b$')plt.show()

### Cómo se ve un modelo que faltaComparemos: los mismos datos, pero generados por una relación con una curvatura leve, ajustados conuna recta.

In [ ]:
xc = np.linspace(1, 10, 15)yc = 2.0*xc + 0.12*xc**2 + np.random.normal(0, 0.4, len(xc))    # hay un término cuadráticopopt_c, _ = curve_fit(recta, xc, yc)R = np.corrcoef(xc, yc)[0, 1]grafico_con_residuos(xc, yc, recta, popt_c,                     xlabel='x', ylabel='y',                     titulo=f'Ajuste lineal de datos con curvatura   —   R = {R:.4f}')plt.show()

$R = 0{,}99$ y pico. En el panel de arriba, el ajuste "se ve bien". En el de abajo, los residuosdibujan una parábola perfectamente visible: **el modelo lineal es incorrecto y los residuos logritan**, mientras que el coeficiente de correlación no dice absolutamente nada al respecto.Ésa es la lección central de este notebook.

---## 3. Qué mide $R$ y qué noEl coeficiente de correlación lineal de Pearson,$$ R = \\frac{\\sum (x_i-\\bar{x})(y_i-\\bar{y})}{\\sqrt{\\sum (x_i-\\bar{x})^2 \\sum (y_i-\\bar{y})^2}} $$mide **cuán bien los datos se alinean sobre una recta**. Eso es todo lo que mide.$R$ **no** mide:- si el modelo que elegiste es el correcto (lo acabás de ver);- causalidad;- si el ajuste es "bueno" en el sentido de compatible con las barras de error — para eso está el  $\\chi^2$ reducido, Colab 06.Y su cuadrado, $R^2$, se interpreta habitualmente como "fracción de varianza explicada". Esainterpretación **solo vale para el ajuste lineal por cuadrados mínimos ordinarios**. Fuera de esecaso pierde sentido, y en modelos no lineales puede ser directamente engañosa (Colab 10).

### El cuarteto de AnscombeCuatro conjuntos de datos publicados por F. J. Anscombe (*The American Statistician* **27**(1),17–21, 1973). Tienen —hasta la segunda o tercera cifra— **el mismo promedio en $x$ y en $y$, la mismavarianza, la misma recta de ajuste y el mismo $R$**.

In [ ]:
x123 = np.array([10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5], float)y1 = np.array([8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68])y2 = np.array([9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74])y3 = np.array([7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73])x4 = np.array([8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8], float)y4 = np.array([6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89])conjuntos = [(x123, y1, 'I'), (x123, y2, 'II'), (x123, y3, 'III'), (x4, y4, 'IV')]print(f"{'conj.':<7}{'x̄':>7}{'ȳ':>8}{'pendiente':>12}{'ordenada':>11}{'R':>9}{'R²':>8}")print("-"*54)for xa, ya, nom in conjuntos:    pa, _ = curve_fit(recta, xa, ya)    r = np.corrcoef(xa, ya)[0, 1]    print(f"{nom:<7}{xa.mean():>7.2f}{ya.mean():>8.2f}{pa[0]:>12.3f}{pa[1]:>11.3f}{r:>9.3f}{r**2:>8.3f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 7))xx = np.linspace(3, 20, 100)for ax, (xa, ya, nom) in zip(axes.ravel(), conjuntos):    pa, _ = curve_fit(recta, xa, ya)    ax.plot(xa, ya, 'o', ms=6)    ax.plot(xx, recta(xx, *pa), 'crimson', lw=1.5)    ax.set_title(f'Conjunto {nom}   (R = {np.corrcoef(xa, ya)[0,1]:.3f})', fontsize=10)    ax.set_xlim(2, 20); ax.set_ylim(2, 14); ax.grid(alpha=0.3)fig.suptitle('Cuarteto de Anscombe: mismos estadísticos, cuatro situaciones distintas', y=1.0)fig.tight_layout(); plt.show()

Leamos los cuatro:- **I** — lo que uno espera: relación lineal con ruido. El ajuste es apropiado.- **II** — la relación es claramente curva. El modelo lineal es incorrecto, y $R$ no se entera.- **III** — relación lineal perfecta arrastrada por **un solo punto atípico**. La pendiente reportada  no describe a los otros diez datos.- **IV** — todos los $x$ valen 8 salvo uno. La "pendiente" está determinada por un único punto: si  ese dato tuviera un error de tipeo, cambiaría todo el resultado.En los cuatro casos, informar "$R = 0{,}82$, el ajuste es bueno" sería igual de defendible — y entres de los cuatro, igual de falso.> **Ejercicio 5.1.** Graficá los residuos de los cuatro conjuntos usando `grafico_con_residuos`.> ¿En cuáles se detecta el problema mirando solamente el panel de residuos?

### Correlación cero no significa independencia$R$ mide relación **lineal**. Puede haber una dependencia funcional exacta y determinista con$R \\approx 0$.

In [ ]:
xs = np.linspace(-3, 3, 200)ys = xs**2                      # dependencia perfecta, sin ruidoprint(f"R entre x y x² (simétrico alrededor de 0) = {np.corrcoef(xs, ys)[0,1]:.2e}")fig, ax = plt.subplots(figsize=(5.5, 3.6))ax.plot(xs, ys, '.', ms=4)ax.set_xlabel('x'); ax.set_ylabel('y = x²')ax.set_title('R ≈ 0 y sin embargo y está completamente determinada por x')ax.grid(alpha=0.3); fig.tight_layout(); plt.show()

---## 4. Linealizar: cuidado con lo que le hacés al errorEs tentador linealizar un modelo para poder usar cuadrados mínimos: tomar logaritmo de unaexponencial, invertir una hipérbola. Es legítimo, pero **la transformación también transforma lasbarras de error**, y casi siempre las vuelve desiguales.Si $y' = \\ln y$, entonces $\\sigma_{y'} = \\sigma_y / y$. Los puntos con $y$ chico pasan a tener unerror relativo enorme, y un ajuste no ponderado sobre los datos linealizados les da el mismo pesoque a los demás. El resultado es un ajuste sesgado.La solución es ajustar en el espacio linealizado **con pesos** (Colab 06) o directamente ajustar elmodelo no lineal (Colab 10).

In [ ]:
# demostración rápida del efectotau_verdadero = 2.0tt = np.linspace(0, 8, 20)yy = 5*np.exp(-tt/tau_verdadero) * np.random.normal(1, 0.08, len(tt))   # 8 % de error relativopl, _ = curve_fit(recta, tt, np.log(yy))            # linealizado, SIN pesostau_lin = -1/pl[0]def expo(t, A, tau):    return A*np.exp(-t/tau)pn, _ = curve_fit(expo, tt, yy, p0=[5, 2])          # no lineal directotau_nl = pn[1]print(f"τ verdadero            : {tau_verdadero:.3f}")print(f"τ por log sin pesos    : {tau_lin:.3f}")print(f"τ por ajuste no lineal : {tau_nl:.3f}")

---## 5. Ejercicios**5.2.** Ajustá tus datos de la ley de Hooke y reportá $k$ con su incerteza, usando `formatear()` delColab 01. Incluí el gráfico con residuos.**5.3.** ¿Qué esperás que valga la ordenada al origen $b$ en el experimento del resorte? Si tu ajusteda un $b$ que difiere de ese valor en más de $2\\sigma_b$, ¿qué podría estar pasando físicamente?**5.4.** Agregá a mano un punto claramente erróneo a tus datos (por ejemplo, moviendo una coma) yvolvé a ajustar. ¿Cuánto cambia $k$? ¿Cuánto cambia $R$? ¿Cuál de los dos te avisó del problema?**5.5.** *(conceptual)* Un compañero informa: "$R^2 = 0{,}9987$, por lo tanto el modelo es correcto".Escribí en tres oraciones por qué esa afirmación no se sostiene, con un contraejemplo del cuarteto.